In [ ]:
import pandas as pd
from glob import glob
import re
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import json
import torch
import numpy as np
from bbox_annotation import Annotation, annotation_from_json
from sklearn.model_selection import train_test_split

from tool import crop_rgb

In [ ]:
np.random.seed(0)

In [ ]:
Image.MAX_IMAGE_PIXELS = None

In [ ]:
image_pattern = re.compile(r"(.*)\.([^\.]*)")

In [ ]:
data = []

In [ ]:
from thing import thing

for i in glob("dataset/images/*"):
    name = Path(i).stem

    annotation_file = Path("dataset/bbox_annotation").joinpath(f"{name}.json")

    try:
        with open(annotation_file, "r") as annotation_file:
            annotation: Annotation = annotation_from_json(json.load(annotation_file))
    except FileNotFoundError as e:
        print(f"{annotation_file} not found, skipping")
        continue

    image = Image.open(i)

    if image.width != annotation["width"] or image.height != annotation["height"]:
        print(f"{annotation_file} has w: {annotation["width"]} h: {annotation["height"]}, but image has dimension w: {annotation["width"]} h: {annotation["height"]}, skipping")
        continue

    data.append({
        "Name": name,
        "Image": i,
        "Image Width": image.width,
        "Image Height": image.height,
        "Depth Column": annotation["bounding_boxs"]["depth_column"],
        "Interpreted Lithology Column": annotation["bounding_boxs"]["interpreted_lithology_column"],
        "Lithology Legend": annotation["bounding_boxs"]["lithology_legend"],
    })
    data.append(thing(Path(i), annotation))

data

In [ ]:
data_df = pd.DataFrame(data)
data_df

In [ ]:
import pickle
with open("dataset.pickle", "wb") as f:
    f.write(pickle.dumps(data_df))

In [ ]:
%store data_df

In [ ]:
training_dataset, rest = train_test_split(data_df, test_size=0.3, random_state=0)
validation_dataset, testing_dataset = train_test_split(rest, test_size=0.5, random_state=0)

display(training_dataset)
display(validation_dataset)
display(testing_dataset)

In [ ]:
data_df.loc[0]

In [ ]:
data_df.loc[0]["Image"]

In [ ]:
crop_rgb(Image.open(data_df.loc[0]["Image"]), data_df.loc[0]["Lithology Legend"])